# 05 — A demanding case: handwritten digits, end to end

This is the capstone. We take everything from this chapter — the network, ReLU, scaling, the loss
curve, the softmax head — to a real **ten-class** problem, and we hold it to the evaluation standards
of chapter 00. The goal is not a victory lap: it is an **honest verdict**, with the limits stated
plainly.

**Prerequisites**
- All of chapter 11 — the neuron, the hidden layer, backpropagation, and the `MLPClassifier` knobs.
- Chapter 00 — a baseline to beat, a held-out test set, the confusion matrix, cross-validation.

**What you'll do**
- Run a full multi-class workflow: look at the data, set baselines, train a scaled MLP, read its loss
  curve, analyse its errors, and check how much its score wobbles with the random seed.
- Compare it *fairly* against tree ensembles, and reach a measured, honest conclusion about when an
  MLP is — and is not — the right tool.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

digits = load_digits()
X, y = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(f"digits: X {X.shape}, {len(np.unique(y))} classes, "
      f"pixel range [{X.min():.0f}, {X.max():.0f}]")
print(f"class counts: {np.bincount(y).tolist()}")
print(f"train {X_tr.shape}   test {X_te.shape}")

## The problem

Each sample is an **8×8 grayscale image** of a handwritten digit — 64 pixel values from 0 to 16 —
and the label is which digit it is, 0 through 9. Ten classes, 1797 images, balanced. It is small and
fast enough to experiment with freely, yet it is a genuine image-recognition task — the kind of raw,
high-dimensional input where "learned features" eventually earn their keep (the bridge to chapter 12).
First, let's look at what we are working with.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for col in range(10):
    idxs = np.where(y == col)[0][:2]
    for row, idx in enumerate(idxs):
        ax = axes[row, col]
        ax.imshow(digits.images[idx], cmap=colors.CMAP_PROBA)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(str(col), color=colors.COLORS["text"])
fig.suptitle("A gallery of handwritten digits (8×8 grayscale, two examples per class)")
plt.show()

**Read the figure.** Two examples of each digit. Most are clear, but notice how much handwriting
varies — some 9s have open loops, some 4s nearly close, some 8s are pinched. A few are ambiguous even
to us. Those are exactly the ones any classifier will struggle with, and we will come back to them.

## A baseline first

Chapter 00's discipline: never report a model's accuracy without a floor to compare it against. Three
references — always guessing the most frequent class, a linear model (logistic regression, scaled),
and a single decision tree.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_tr, y_tr)

print(f"Dummy (most frequent) : {dummy.score(X_te, y_te):.3f}")
print(f"Logistic (scaled)     : {logistic.score(X_te, y_te):.3f}")
print(f"Single decision tree  : {tree.score(X_te, y_te):.3f}")

**Read the result.** Guessing the most frequent class scores 0.10 — chance, for ten balanced classes.
A single decision tree reaches 0.86. And a *linear* model already hits 0.97: after scaling, the digits
are very nearly linearly separable, so the bar our network has to clear is genuinely high. That is
worth remembering before we get excited about any neural-network number.

## The multi-layer perceptron, scaled

We wrap the network in a `StandardScaler` pipeline — scaling is mandatory for an MLP (notebook 11.4),
and the pixel intensities, while bounded, still benefit. Then we train and watch the **loss curve**
to confirm it converged (chapter 03 / notebook 11.4).

In [ ]:
def scaled_mlp(seed=SEED, **kwargs):
    return make_pipeline(StandardScaler(), MLPClassifier(random_state=seed, **kwargs))


mlp = scaled_mlp(hidden_layer_sizes=(100,), max_iter=300).fit(X_tr, y_tr)
clf = mlp.named_steps["mlpclassifier"]
test_acc = mlp.score(X_te, y_te)
print(f"scaled MLP(100,): test accuracy {test_acc:.4f}   (converged in {clf.n_iter_} epochs)")

C = colors.COLORS
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(clf.loss_curve_, color=C["model"], lw=2)
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title(f"MLP training loss on digits (test acc {test_acc:.3f}, {clf.n_iter_} epochs)")
plt.show()

**Read the figure.** The loss falls from about 2.5 (ten classes, no idea yet) to near zero over ~145
epochs, then flattens — the network converged well inside the `max_iter` budget. Its sealed-test
accuracy is 0.978, comfortably above every baseline. So far, so good — but accuracy is a single number,
and a capstone owes you more than that.

## Where does it go wrong?

A single accuracy hides the *structure* of the mistakes. The **confusion matrix** shows, for each true
digit, what the model predicted — the diagonal is correct, everything off it is an error.

In [ ]:
y_pred = mlp.predict(X_te)
cm = confusion_matrix(y_te, y_pred)
viz.plot_confusion_matrix(cm, class_names=[str(d) for d in range(10)])
plt.show()

n_err = int(cm.sum() - np.trace(cm))
print(f"test errors: {n_err} / {len(y_te)}   (accuracy {1 - n_err / len(y_te):.4f})")

**Read the figure.** Almost everything sits on the diagonal: only 10 of 450 test digits are
misclassified. The few off-diagonal cells are the telling ones — a couple of 9s read as 3s, an 8 as a
3 or a 1 — confusions between digits that really do share strokes. No class collapses; the errors are
scattered and sensible.

In [ ]:
errors = np.where(y_pred != y_te)[0]
fig, axes = plt.subplots(1, len(errors), figsize=(1.5 * len(errors), 2.2))
for ax, idx in zip(axes, errors, strict=True):
    ax.imshow(X_te[idx].reshape(8, 8), cmap=colors.CMAP_PROBA)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{y_te[idx]} → {y_pred[idx]}", color=colors.COLORS["error"], fontsize=10)
fig.suptitle(f"All {len(errors)} misclassified test digits (true → predicted)")
plt.show()

**Read the figure.** Here is every single mistake. Look at them: these are genuinely ambiguous
strokes — a 9 whose loop did not close, an 8 squeezed thin, a 4 with a gap. A human would hesitate on
several of them too. The model is not failing in some systematic, fixable way; it is missing the
hardest, blurriest cases. That is an honest place for the errors to live.

## Is 0.978 reliable?

One number from one training run can mislead. A multi-layer network's loss is **non-convex**
(notebook 11.3): different random initializations settle in different minima, so the same model on the
same data can land on slightly different accuracies. The honest move is to train it several times and
report the *spread*.

In [ ]:
seed_accs = np.array([
    scaled_mlp(seed=s, hidden_layer_sizes=(100,), max_iter=300).fit(X_tr, y_tr).score(X_te, y_te)
    for s in range(10)
])
print(f"test accuracy over 10 seeds: min {seed_accs.min():.4f}  max {seed_accs.max():.4f}  "
      f"mean {seed_accs.mean():.4f}  std {seed_accs.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.scatter(seed_accs, np.zeros_like(seed_accs), s=130, color=C["model"],
           edgecolor=C["text"], alpha=0.8, zorder=3)
ax.axvline(seed_accs.mean(), color=C["highlight"], ls="--", lw=2,
           label=f"mean {seed_accs.mean():.3f}")
ax.set_yticks([])
ax.set_xlabel("sealed-test accuracy")
ax.set_title("Same MLP, 10 random initializations: the accuracy spread")
ax.legend()
plt.show()

**Read the figure.** Across ten seeds the accuracy ranges from about 0.971 to 0.982 — a swing of
roughly half a percent from the random initialization *alone*, nothing else changed. So 0.978 is not
a single sharp truth; it is a draw from a narrow band. Whenever you report a neural network's score,
report this spread too, and never crown a winner on a difference smaller than it.

## Is the MLP the right tool here?

The honest question is not "is the MLP good?" (it is) but "is it the *best choice* for this problem?"
To answer fairly we compare it with tree ensembles — and we give each method what it needs: the MLP
its `StandardScaler`, the trees their **raw** features (trees are scale-invariant, so scaling them is
pointless). We also stop trusting a single split: a 450-point test set wobbles by a percent, so we
compare on **5-fold cross-validation** with error bars.

In [ ]:
# Light tuning, judged once on the sealed test (the grid is scored by CV on the training data).
param_grid = {
    "mlpclassifier__hidden_layer_sizes": [(64,), (100,), (128,)],
    "mlpclassifier__alpha": [1e-4, 1e-3, 1e-2],
}
search = GridSearchCV(scaled_mlp(max_iter=300), param_grid, cv=4, n_jobs=-1).fit(X_tr, y_tr)
print(f"default MLP sealed test {test_acc:.4f}   tuned sealed test {search.score(X_te, y_te):.4f}")
print(f"best params {search.best_params_}")

# Fair cross-method comparison: 5-fold CV, MLP scaled vs trees on raw features.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def cv_mean_std(estimator, data_X, data_y):
    scores = cross_val_score(estimator, data_X, data_y, cv=cv)
    return scores.mean(), scores.std()


digit_methods = {
    "MLP": scaled_mlp(hidden_layer_sizes=(100,), max_iter=300),
    "RF": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(random_state=SEED),
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
}
digit_cv = {name: cv_mean_std(est, X, y) for name, est in digit_methods.items()}
print()
print("digits, 5-fold CV accuracy (MLP/Logistic scaled, trees raw):")
for name, (mean, std) in digit_cv.items():
    print(f"  {name:9s}: {mean:.4f} +/- {std:.4f}")

# breast_cancer: clean, homogeneous tabular data -- where the MLP leads.
Xb, yb = load_breast_cancer(return_X_y=True)
bc_methods = {
    "MLP": scaled_mlp(hidden_layer_sizes=(32,), max_iter=2000),
    "RF": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(random_state=SEED),
}
bc_cv = {name: cv_mean_std(est, Xb, yb) for name, est in bc_methods.items()}
print()
print("breast_cancer, 5-fold CV accuracy:")
for name, (mean, std) in bc_cv.items():
    print(f"  {name:9s}: {mean:.4f} +/- {std:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

d_names = list(digit_cv)
d_means = [digit_cv[n][0] for n in d_names]
d_stds = [digit_cv[n][1] for n in d_names]
axes[0].bar(d_names, d_means, yerr=d_stds, capsize=4, edgecolor=C["text"],
            color=[C["model"], C["class_c"], C["class_b"], C["muted"]])
axes[0].set_ylim(0.95, 1.0)
axes[0].set_ylabel("5-fold CV accuracy")
axes[0].set_title("digits — MLP (scaled) vs trees (raw): a tie")

b_names = list(bc_cv)
b_means = [bc_cv[n][0] for n in b_names]
b_stds = [bc_cv[n][1] for n in b_names]
axes[1].bar(b_names, b_means, yerr=b_stds, capsize=4, edgecolor=C["text"],
            color=[C["model"], C["class_c"], C["class_b"]])
axes[1].set_ylim(0.93, 1.0)
axes[1].set_ylabel("5-fold CV accuracy")
axes[1].set_title("breast_cancer — the MLP leads")
plt.show()

**Read the figure.** On the digits (left) the MLP, random forest, and histogram-boosted trees land
within a whisker of each other — about 0.973–0.977, error bars overlapping. **It is a tie:** the MLP
is fully competitive but not meaningfully ahead, and the trees got there on raw features with no
scaling and stay far more interpretable. On `breast_cancer` (right) the MLP **leads** — 0.975, ahead
of the random forest (0.958) and a touch ahead of HistGB (0.970). On this clean, homogeneous
tabular data the network's flexibility helps where it did not on the digits. Same model family, two
datasets, two verdicts: there is **no universal best**; the right tool depends on the problem.

## The honest limits

The capstone earns its name by stating what the MLP costs you, not only what it scores:

- **Non-convex and seed-sensitive.** Gradient descent finds *a* good minimum, not *the* minimum
  (Figure 5's spread). Always report the variability.
- **Universal approximation is an existence result.** A big enough network *can* represent almost any
  boundary (notebook 11.2) — but "can represent" is not "will learn", and certainly not "will beat a
  tree". The digits tie is the proof.
- **Not interpretable.** A decision tree or logistic weights can be read; this network's 7,400 weights
  cannot.
- **Scaling is mandatory**, and tuning bought us nothing here — the defaults were already well matched.
- **So why neural networks?** Their edge is *learned features* on raw, high-dimensional inputs —
  images, audio, text — where hand-engineering features is hopeless. That is the conceptual promise
  this 8×8 problem only hints at, and the subject of chapter 12.

## Your turn

1. **(warm-up)** Change the MLP's `random_state` and re-run the error gallery. Does the *set* of
   misclassified digits change? Connect what you see to the non-convexity from notebook 11.3.
2. **(core)** Try `hidden_layer_sizes=(100, 100)` or a different `alpha`. Does the sealed-test accuracy
   move *beyond* the ±0.005 seed band from Figure 5? Would you trust a 0.001 "improvement"?
3. **(reach)** Flip 2% of the *training* labels at random (label noise) and re-fit both the MLP and the
   random forest. Which one degrades more gracefully, and why might that be?

## What you built — the whole chapter

- A single sigmoid neuron *is* logistic regression (11.1).
- A hidden layer plus a non-linearity carves boundaries one neuron cannot — and a linear stack
  collapses (11.2).
- Backpropagation is the chain rule, layer by layer; you derived it, gradient-checked it, and trained
  with it (11.3).
- `MLPClassifier` is that machine with dials: ReLU, the solver, the loss curve, capacity, `alpha`,
  scaling, and the K-class softmax head (11.4).
- And here you ran a full, honest workflow and reached a measured verdict (11.5).

You can now build, train, tune, evaluate, **and honestly judge** a neural network.

## Where chapter 12 goes

Chapter 11 was "the MLP as scikit-learn ships it": one hidden layer, parameter-space gradient descent,
a CPU-trainable estimator. Chapter 12 takes on neural networks as a paradigm — **depth as a hierarchy
of learned representations**, dropout, the gradient pathologies that depth creates, modern
initialization and normalization, and the move to a deep-learning framework — where the "learned
features" promise finally pays off.

## References

- LeCun, Y., Bottou, L., Bengio, Y. & Haffner, P. (1998). *Gradient-based learning applied to document
  recognition.* Proc. IEEE 86(11), 2278–2324. https://doi.org/10.1109/5.726791 (the digit-recognition
  lineage).
- Grinsztajn, L., Oyallon, E. & Varoquaux, G. (2022). *Why do tree-based models still outperform deep
  learning on tabular data?* https://arxiv.org/abs/2207.08815
- Shwartz-Ziv, R. & Armon, A. (2022). *Tabular data: Deep learning is not all you need.* Information
  Fusion 81, 84–90. https://doi.org/10.1016/j.inffus.2021.11.011
- Cybenko 1989 / Hornik 1991 — universal approximation (existence). scikit-learn user guide:
  `load_digits`, `MLPClassifier`.

*Previous:* 11.4 — the estimator & its parameters.  *Next:* **Chapter 12 — Neural Networks.**